#### The following notebook outlines a subset of the pipelines used in the Lichess big data analysis project

In [ ]:
#import section
import pymongo
from pymongo import MongoClient
from datetime import datetime
from pprint import pprint
import networkx as nx
import pandas as pd
import re

# Creation of pyMongo connection object
client = MongoClient('mongodb://localhost:27017')
# games.zip includes the games collection you need to use
db = client['coursework']
games_collection = db['games']

### Extracting Player Win Streaks

In [ ]:
def winning_streak(games_collection, timeControl, minGames):
    pipeline = [
        {
            "$match": {
                "TimeControl": timeControl
            }
        },
        {
            "$sort": {
                "UTCdate": 1,
                "UTCtime": 1
            }
        },
        {
            "$project": {
                "player": {
                    "$cond": {
                        "if": { "$eq": ["$result", "1-0"] },
                        "then": "$White",
                        "else": {
                            "$cond": {
                                "if": { "$eq": ["$result", "0-1"] },
                                "then": "$Black",
                                "else": None
                            }
                        }
                    }
                },
                "result": {
                    "$cond": {
                        "if": { "$eq": ["$result", "1-0"] },
                        "then": "Win",
                        "else": {
                            "$cond": {
                                "if": { "$eq": ["$result", "0-1"] },
                                "then": "Lose",
                                "else": "Draw"
                            }
                        }
                    }
                }
            }
        },
        {
            "$group": {
                "_id": "$player",
                "results": { "$push": "$result" },
                "totalGames": { "$sum": 1 }
            }
        },
        {
            "$match": {
                "totalGames": { "$gte": minGames }
            }
        }
    ]

    result = list(games_collection.aggregate(pipeline))

    
    players = []
    for player_data in result:
        player = player_data['_id']
        total_games = player_data['totalGames']
        results = player_data['results']
        
        
        max_streak = 0
        current_streak = 0
        for result in results:
            if result == 'Win':
                current_streak += 1
                if current_streak > max_streak:
                    max_streak = current_streak
            else:
                current_streak = 0
        
       
        players.append((player, max_streak))


    players_df = pd.DataFrame(players, columns=['username', 'winning_streak'])


    players_df = players_df.sort_values(by=['winning_streak', 'username'], ascending=[False, False])
    players_df.head
    return players_df
        

In [ ]:
%%time
answer = winning_streak(games_collection,timeControl="300+0",minGames=3)
pprint(answer)

### Using Hadoop's MapReduce Paradigm to count the average number of checks made with a certain piece

In [ ]:
%%writefile mapper.py
#!/usr/bin/env python
import sys
import json


for line in sys.stdin:
    jsonData = json.loads(line.strip())
    moves = jsonData.get('moves')
    print('Total\t1')
    for move in moves:
        move_type = move['move']
        if move_type[-1] == '+':
            if move_type[:3] == 'O-O' or move_type[:5] == 'O-O-O':
                piece = 'King'
            else:
                first_letter = move_type[0]
                if first_letter == 'N':
                    piece = 'Knight'
                elif first_letter == 'B':
                    piece = 'Bishop'
                elif first_letter == 'R':
                    piece = 'Rook'
                elif first_letter == 'Q':
                    piece = 'Queen'
                elif first_letter == 'K':
                    piece = 'King'
                else:
                    piece = 'Pawn'
            print(piece + '\t' + '1')


In [ ]:
%%writefile reducer.py
#!/usr/bin/env python
import sys

accumulator = {}
total_games = 0

for line in sys.stdin:
    key, value = line.split('\t')
    value = int(value)
    if key == 'Total':
        total_games = total_games + value
    else:
        accumulator[key] = accumulator.get(key, 0) + value


for piece in accumulator.keys():
    average = accumulator[piece] / total_games 
    print(f'{piece}: {average}')

In [ ]:
%%time
%%bash
#Hadoop command to run the map reduce.
#In case of error, check file /tmp/q11_errors.log
rm -rf output
mapred streaming \
-files mapper.py,reducer.py   \
-input /home/games.json   \
-mapper mapper.py  \
-reducer reducer.py  \
-output output 2> /tmp/q11_errors.log
